# 51. 원본을 한 번 읽으면 MPS 오버레이 완성이 빨라질까?

50번과 같은 검증 영상 c04 116장을 사용합니다. MPS, FP32, 입력 긴 변 640, 배치 1, 1080×1920 H.264 출력을 고정합니다. `two-pass`는 기존 경로입니다. `single-pass`는 한 장에서 원판·자세 모델을 차례로 호출하고, `single-shared`는 한 장의 리사이즈·정규화·GPU 입력 텐서도 두 모델이 공유합니다. 한 번 읽는 경로는 원판 궤적을 프레임마다 연결합니다.

각 방식은 새 프로세스에서 세 번 실행합니다. 한 블록에서 세 방식 모두 실행하고 시작 순서를 바꿉니다. 모델 준비부터 완성 파일 저장까지와 명령 시작부터 종료까지를 별도로 기록합니다. 프로젝트 Python 커널에서 위부터 실행하세요.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))

from wlcv.evaluation.overlay_read_timing import MODES, timed_runs, verify_runs
import matplotlib.pyplot as plt

folder = root / "outputs/overlay_read_comparison" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
runs = timed_runs(folder)
display(runs[["block", "mode", "command_s", "pipeline_s"]].round(3))

## 같은 프레임과 완성 영상인지 확인

첫 두 번 읽기 실행을 기준으로 9개 MP4를 끝까지 다시 엽니다. 복원한 픽셀, 원판 검출·연결·미탐 구분, 사람·관절 수, 좌표·점수·체간 각도를 대조합니다. 이 시간은 벤치마크에 포함하지 않습니다.

In [ ]:
checks = verify_runs(runs, folder)
display(checks)
print("실행 결과 폴더", folder)
print("한 번 읽고 전처리까지 공유한 영상", runs.loc[runs["mode"].eq("single-shared"), "run"].iloc[-1])

## 영상 한 편의 시간

점 하나는 같은 116장을 처음부터 끝까지 처리한 한 번의 명령입니다. 같은 순번 블록 안에서 시간을 대조하고, 세 실행의 중앙값을 비교합니다.

In [ ]:
by_block = runs.pivot(index="block", columns="mode", values="command_s")[list(MODES)]
display(by_block.round(3))
medians = runs.groupby("mode")[["command_s", "pipeline_s"]].median().reindex(MODES)
display(medians.round(3))
ax = by_block.T.plot(marker="o", figsize=(7, 3.6))
ax.set(xlabel="Read mode", ylabel="Command wall time (s)")
ax.set_xticks(range(len(MODES)), MODES)
ax.figure.tight_layout()
ax.figure.savefig(folder / "command_times.png", dpi=130)
plt.show()